# Chapter 9: lineage from grain lot to pallet

The book prints the forward trace as a recursive common table expression (sql/lineage_forward.sql) and the
Fabric Warehouse form as a T-SQL WHILE loop (sql/lineage_forward_loop.sql). This notebook runs the CTE on
SQLite, runs the same loop in Python (the notebook form) and checks the two agree.

Dialects: SQLite and PostgreSQL run the CTE as printed. SQL database in Fabric runs it with the word RECURSIVE
dropped. The Fabric Warehouse and the SQL analytics endpoint cannot run a recursive CTE; use the loop there.
The T-SQL loop file does not run on SQLite (no temporary #tables, DECLARE or WHILE), which is why the Python
loop below stands in for it here. A trace nominates candidates. QA decides holds, releases and recalls.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import kestrel as k

con = k.connect()
edges = k.q(con, "SELECT * FROM edge")
print(edges[edges.material_class.isin(["malt", "liquid"])].head(12).to_string(index=False))

In [ ]:
cte = k.q(con, (k.SQL / "lineage_forward.sql").read_text()).set_index("node").share
loop = k.trace(edges, "M-26-0087")
assert set(cte.index) == set(loop.index) and (cte - loop[cte.index]).abs().max() < 1e-12
print("CTE and loop agree on", len(cte), "nodes")
print(cte[~cte.index.str.startswith("0")].round(3).to_string())

The pallets, with where each one is now. A pallet is shipped when a ship movement leaves it.

In [ ]:
pallets = k.q(con, """
    SELECT p.sscc, l.lot_id AS pack_lot, p.volume_hl,
           EXISTS (SELECT 1 FROM movement m WHERE m.movement_type = 'ship' AND m.from_node_type = 'pallet'
                   AND m.from_node_key = p.package_unit_key) AS shipped
    FROM package_unit p JOIN lot l ON l.lot_key = p.pack_lot_key""")
hit = pallets[pallets.sscc.isin(cte.index)].copy()
hit["share"] = hit.sscc.map(cte)
table = hit.groupby("pack_lot").agg(pallets=("sscc", "count"), beer_hl=("volume_hl", "sum"),
                                    share=("share", "first"), shipped=("shipped", "sum"))
table["in_warehouse"] = table.pallets - table.shipped
left = edges[edges.from_node == "F07-26-034"].quantity.sum() - table.beer_hl.sum()
print(table.round(3).to_string())
print(f"still in bright tank BT02-26-031: {left:.2f} hL")
print(f"closure: {table.beer_hl.sum():.2f} + {left:.2f} = {table.beer_hl.sum() + left:.2f} hL left the fermenter")

Exposure: share times quantity, on the hL edges only. Never add kilograms of malt to hectolitres of beer.

In [ ]:
share = round(cte["BT02-26-031"], 3)  # multiply by the share as printed
for name, hl in list(table.beer_hl.items()) + [("in tank", left)]:
    print(f"{name:14s} {hl:7.2f} hL x {share:.3f} = {hl * share:6.2f} hL")
print(f"total {(table.beer_hl.sum() + left) * share:.1f} hL")

Backward from one can pallet: swap from and to (sql/lineage_backward.sql). Filter to malt to answer the question
asked; keep the full list for the day the cause is a can lot.

In [ ]:
back = k.q(con, (k.SQL / "lineage_backward.sql").read_text()).set_index("node").share
back_loop = k.trace(edges, "006141412600000416", backward=True)
assert (back - back_loop[back.index]).abs().max() < 1e-12
print(back[back.index.str.startswith("M-")].round(3).to_string())
print(back[back.index.str[0].isin(list("HYGC"))].round(3).to_string())